# Regenerate Table 8 (clutter) and the shape-control column of Table 9 on the final split

Reviewer request: both were from earlier runs. This notebook rebuilds them on the **final group-aware split**
with the **same five 70-label probes** as every other result (seed `rng(42*1000+rep)`, multinomial logistic, C = 1).

* **Table 9, shape column** (CPU only): seven leaf-geometry descriptors per image (relative area, aspect ratio,
  solidity, circularity, perimeter-to-size, eccentricity, fill fraction) from the same Otsu leaf mask;
  the same labelled leaves as the feature probe at every budget; same classifier with standardisation; AUC of
  1 − P(Healthy) on the 3,929 test images.
* **Table 8, clutter** (GPU recommended, CPU works): the original 40 synthetic field textures, blended over the
  **whole full-resolution image** at α ∈ {0, 0.2, 0.4, 0.6}, then the usual mask → crop → JPEG q95 → DINO.
  α = 0 must reproduce the published probe AUC 0.985 (the notebook stops if it does not).

Output: `results_revision_v1/regen_tables_8_9.json` — upload it and I will update the tables and text.

Runtime on Colab: ~10 min with a GPU (shape part ~3 min on CPU).

In [ ]:
# CELL 0 — CONFIG (same paths as revision_v1_reproduce.ipynb)
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
OUT        = f'{RES}/regen_tables_8_9.json'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
SHOTS  = (1, 2, 5, 10, 20, 50, 100)
ALPHAS = (0.0, 0.2, 0.4, 0.6)


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
Xtr, Xte = F['Xtr'], F['Xte']
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


# --- array versions (same code path as leaf_crop, used for composited images) ---
def mask_arr(rgb):
    if _ORIG:
        try:
            try: return np.asarray(_sc.leaf_mask(rgb)).astype(bool)
            except Exception: return np.asarray(_sc.leaf_mask(Image.fromarray(rgb))).astype(bool)
        except Exception: pass
    return _fb_mask(rgb)

def leaf_crop_arr(rgb):
    if _ORIG:
        try:
            m = mask_arr(rgb)
            im = _fsc.crop_to_leaf(rgb, m)
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))


In [ ]:
# CELL 5 — leaf-geometry descriptors for every image (CPU, parallel)
def _fb_shape(m):
    m8 = m.astype(np.uint8); cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea); a = float(cv2.contourArea(c)); p = float(cv2.arcLength(c, True))
    if a < 1 or p < 1: return None
    x, y, w, h = cv2.boundingRect(c); ha = float(cv2.contourArea(cv2.convexHull(c))) + 1e-6
    if len(c) >= 5:
        (_, _), (MA, ma), _ = cv2.fitEllipse(c); ecc = float(np.sqrt(1 - (min(MA, ma) / (max(MA, ma) + 1e-6)) ** 2))
    else: ecc = 0.0
    return np.array([a / m.size, w / (h + 1e-6), a / ha, 4 * np.pi * a / (p ** 2 + 1e-6),
                     p / (np.sqrt(a) + 1e-6), ecc, float(m.sum()) / m.size], np.float32)

def shape_of(rec):
    try:
        img = Image.open(rec['path']).convert('RGB')
        if _ORIG:
            m = _sc.leaf_mask(img)                      # original: mask at 224x224, holes filled
            f = None if m is None else _sc.shape_features(m)
        else:
            m = _fb_mask(np.array(img.resize((IMG, IMG)))); f = _fb_shape(m)
        return None if f is None else np.asarray(f, np.float32)
    except Exception:
        return None

t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    S_tr = list(ex.map(shape_of, TRAIN, chunksize=32)); S_te = list(ex.map(shape_of, TEST, chunksize=32))
nfail = sum(s is None for s in S_tr + S_te)
dim = next(len(s) for s in S_tr if s is not None)
fill = lambda L: np.stack([s if s is not None else np.full(dim, np.nan, np.float32) for s in L])
S_tr, S_te = fill(S_tr), fill(S_te)
med = np.nanmedian(S_tr, 0)                                  # impute the rare failures with the train median
S_tr = np.where(np.isnan(S_tr), med, S_tr); S_te = np.where(np.isnan(S_te), med, S_te)
print('shape descriptors: %d dims, %d failures of %d, %.0fs' % (dim, nfail, len(TRAIN) + len(TEST), time.time() - t0))


In [ ]:
# CELL 6 — Table 9 shape-control column: same labelled leaves, same classifier family, same score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
def draw(rep, n):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def shape_probe(idx):
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000)).fit(S_tr[idx], ytr[idx])
T9 = {}
for n in SHOTS:
    a = [roc_auc_score(dte, 1 - shape_probe(draw(rep, n)).predict_proba(S_te)[:, HEALTHY]) for rep in range(5)]
    T9[str(n)] = dict(mean=float(np.mean(a)), sd=float(np.std(a, ddof=1)), per_rep=[float(x) for x in a])
    print('shape control, %3d per class: AUC %.3f +- %.3f' % (n, np.mean(a), np.std(a, ddof=1)))
allp = shape_probe(np.arange(len(TRAIN)))
T9['all'] = dict(mean=float(roc_auc_score(dte, 1 - allp.predict_proba(S_te)[:, HEALTHY])))
print('shape control, all training labels: AUC %.3f' % T9['all']['mean'])
# reference: the feature probe with the same draws must reproduce the published curve
fp = [roc_auc_score(dte, 1 - LogisticRegression(C=1.0, max_iter=2000).fit(Xtr[draw(r, 10)], ytr[draw(r, 10)]).predict_proba(Xte)[:, HEALTHY]) for r in range(5)]
print('feature probe, 10 per class (must be 0.985):', round(float(np.mean(fp)), 4))
assert abs(np.mean(fp) - 0.9853) < 1e-3, 'draws differ from the published run: stop'


In [ ]:
# CELL 7 — DINO ViT-S/16 and the original synthetic field textures
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device).eval()
@torch.no_grad()
def feats_pil(ims, bs=128):
    out = []
    for i in range(0, len(ims), bs):
        x = torch.stack([TO_T(im) for im in ims[i:i + bs]]).to(device)
        out.append(torch.nn.functional.normalize(dino(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out)

def field_textures(n=40, size=224, seed=0):          # verbatim from scripts/original/clutter_robustness_fast.py
    rng = np.random.RandomState(seed); out = []
    palettes = [[(34, 85, 34), (60, 110, 45), (20, 60, 25)], [(85, 65, 34), (110, 90, 55), (60, 45, 25)],
                [(45, 90, 45), (70, 120, 60), (30, 70, 35)], [(70, 60, 40), (95, 85, 60), (50, 40, 30)]]
    for i in range(n):
        pal = palettes[i % len(palettes)]; img = np.zeros((size, size, 3), np.uint8); img[:] = pal[0]
        for _ in range(70):
            x, y = rng.randint(0, size - 20, 2); w, h = rng.randint(15, 70, 2)
            col = np.clip(np.array(pal[rng.randint(len(pal))]) + rng.randint(-30, 30, 3), 0, 255)
            img[y:y + h, x:x + w] = col
        out.append(cv2.GaussianBlur(img, (9, 9), 3).astype(np.float32))
    return np.stack(out)
BGS = field_textures()
BG_CHOICE = np.random.default_rng(0).integers(0, len(BGS), len(TEST))   # one texture per test leaf, fixed across alpha


In [ ]:
# CELL 8 — composite at full resolution, crop exactly as the cache, JPEG q95, features
def composite_crop(args):
    i, alpha = args
    rgb = np.array(Image.open(TEST[i]['path']).convert('RGB'))
    if alpha > 0:
        bg = cv2.resize(BGS[BG_CHOICE[i]], (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        rgb = ((1 - alpha) * rgb.astype(np.float32) + alpha * bg).clip(0, 255).astype(np.uint8)
    try: im = leaf_crop_arr(rgb)
    except Exception: return i, None
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95); buf.seek(0)   # same JPEG round trip as the crop cache
    return i, Image.open(buf).convert('RGB')

XC, SEGFAIL = {}, {}
for a in ALPHAS:
    t0 = time.time()
    with ThreadPoolExecutor(os.cpu_count()) as ex:
        res = dict(ex.map(composite_crop, [(i, a) for i in range(len(TEST))]))
    bad = [i for i, im in res.items() if im is None]
    ims = [res[i] if res[i] is not None else Image.new('RGB', (IMG, IMG)) for i in range(len(TEST))]
    XC[a], SEGFAIL[a] = feats_pil(ims), len(bad)
    print('alpha %.1f: features %s, crop failures %d, %.0fs' % (a, XC[a].shape, len(bad), time.time() - t0))
cos0 = (XC[0.0] * Xte).sum(1)
print('alpha 0 vs cached test features: median cosine %.4f' % np.median(cos0))
assert np.median(cos0) > 0.99, 'alpha=0 crops do not reproduce the cache: stop'


In [ ]:
# CELL 9 — Table 8: the five published 70-label probes, evaluated on composited test leaves
T8 = {}
for a in ALPHAS:
    auc, acc = [], []
    for rep in range(5):
        idx = draw(rep, N_SHOT); p = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr[idx], ytr[idx])
        P = p.predict_proba(XC[a]); auc.append(roc_auc_score(dte, 1 - P[:, HEALTHY]))
        acc.append(accuracy_score(dte, (P.argmax(1) != HEALTHY).astype(int)))   # same rule as Table 9
    T8[str(a)] = dict(auc=float(np.mean(auc)), auc_sd=float(np.std(auc, ddof=1)), acc=float(np.mean(acc)),
                      acc_sd=float(np.std(acc, ddof=1)), crop_failures=SEGFAIL[a], per_rep_auc=[float(x) for x in auc])
    print('alpha %.1f: AUC %.3f +- %.3f | binary acc %.3f +- %.3f' % (a, np.mean(auc), np.std(auc, ddof=1), np.mean(acc), np.std(acc, ddof=1)))
assert abs(T8['0.0']['auc'] - 0.9853) < 2e-3, 'clean row does not reproduce 0.985: stop'

meta = dict(split='final group-aware split, 3,929 test images', probe='multinomial logistic C=1, draws rng(42*1000+rep), 5 reps',
            shape='7 geometry descriptors from the Otsu leaf mask, StandardScaler + same classifier, score 1-P(Healthy)',
            clutter='40 synthetic field textures (scripts/original/clutter_robustness_fast.py), blended over the full-resolution image, then mask, crop, JPEG q95, DINO',
            original_modules=_ORIG)
json.dump(dict(table9_shape=T9, table8_clutter=T8, _meta=meta), open(OUT, 'w'), indent=1)
print('saved', OUT, '-> upload this file')
